# Imports 

In [38]:
import pandas as pd
import numpy as np
import os

import seaborn

## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Read in data and make sure relevant columns are string/character 

- San Diego data: `naics_code` and `account_key`
- NAICS details data: `naics` (North American Industry Classification Systems)

Run code below; if pulling from github, pathname should be fine; if working elsewhere may need to edit path name at read in 

In [39]:
# uncomment if u need to find your path: os.getcwd()
SD = sd_df = pd.read_csv("../public_data/sd_df.csv")
NA = naics_df = pd.read_csv("../public_data/naics_df.csv")

In [40]:
cols_sd_use = ["naics_code", "account_key"]
cols_naics_use = ["naics"]

sd_df[cols_sd_use] = sd_df[cols_sd_use].astype(str)
naics_df[cols_naics_use] = naics_df[cols_naics_use].astype(str)

sd_df.dtypes
naics_df.dtypes
sd_df.head()
naics_df.head()

account_key          object
dba_name             object
council_district     object
naics_code           object
naics_description    object
naics_nchar           int64
dtype: object

naics                object
naics_description    object
dtype: object

,account_key,dba_name,council_district,naics_code,naics_description,naics_nchar
0,1974000448,ERNST & YOUNG LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,cd_3,5411,LEGAL SERVICES,4
2,1978039819,RSM US LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,cd_3,5411,LEGAL SERVICES,4
4,1979046817,KORENIC & WOJDOWSKI LLP,cd_7,5412,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,4


,naics,naics_description
0,111140,Wheat Farming
1,111160,Rice Farming
2,111150,Corn Farming
3,111110,Soybean Farming
4,111120,Oilseed (except Soybean) Farming


In [10]:
sd_df.head()

,account_key,dba_name,council_district,naics_code,naics_description,naics_nchar
0,1974000448,ERNST & YOUNG LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,cd_3,5411,LEGAL SERVICES,4
2,1978039819,RSM US LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,cd_3,5411,LEGAL SERVICES,4
4,1979046817,KORENIC & WOJDOWSKI LLP,cd_7,5412,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,4


In [9]:
naics_df.head()

,naics,naics_description
0,111140,Wheat Farming
1,111160,Rice Farming
2,111150,Corn Farming
3,111110,Soybean Farming
4,111120,Oilseed (except Soybean) Farming


## "Inner join"- retain only San Diego businesses with details on their NAICS code

- Use the `naics_code` column in the San Diego business data as the join key
- Use the `naics` column in the NAICS code details data as the join key

- Do an inner join of the San Diego data onto the NAICS code details using these join keys
- After the inner join, print some examples of San Diego businesses lost in the merge
- Use value_counts() on the `naics_nchar` column in the San Diego data to see why they might have gotten lost


In [41]:
merged_df = naics_df.merge(sd_df, left_on="naics", right_on="naics_code", how="inner")
merged_df.head()

,naics,naics_description_x,account_key,dba_name,council_district,naics_code,naics_description_y,naics_nchar
0,311999,All Other Miscellaneous Food Manufacturing,2020007786,PRIMAL COFFEE COMPANY,cd_2,311999,ALL OTHER MISCELLANEOUS FOOD MFG,6
1,311999,All Other Miscellaneous Food Manufacturing,2020007786,PRIMAL COFFEE COMPANY,cd_2,311999,ALL OTHER MISCELLANEOUS FOOD MFG,6
2,311999,All Other Miscellaneous Food Manufacturing,2020007786,PRIMAL COFFEE COMPANY,cd_2,311999,ALL OTHER MISCELLANEOUS FOOD MFG,6
3,311999,All Other Miscellaneous Food Manufacturing,2020007786,PRIMAL COFFEE COMPANY,cd_2,311999,ALL OTHER MISCELLANEOUS FOOD MFG,6
4,311999,All Other Miscellaneous Food Manufacturing,2020007786,PRIMAL COFFEE COMPANY,cd_2,311999,ALL OTHER MISCELLANEOUS FOOD MFG,6


In [42]:
lost_businesses = sd_df[~sd_df["naics_code"].isin(merged_df["naics_code"])]
lost_businesses.head()

,account_key,dba_name,council_district,naics_code,naics_description,naics_nchar
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,cd_3,5411,LEGAL SERVICES,4
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,cd_3,5411,LEGAL SERVICES,4
4,1979046817,KORENIC & WOJDOWSKI LLP,cd_7,5412,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,4
5,1979053082,GRIMM VRANJES GREER STEPHAN & BRIDGMAN LLP,cd_3,5411,LEGAL SERVICES,4
6,1981000840,BENINK & SLAVENS LLP,cd_7,5411,LEGAL SERVICES,4


In [19]:
sd_df["naics_nchar"].value_counts()

naics_nchar
5    150
4     89
6     49
3     42
2     23
Name: count, dtype: int64

In [18]:
merged_df["naics_nchar"].value_counts()

naics_nchar
6    52
Name: count, dtype: int64

## "Left join"- retain all sd businesses even if naics code isn't in `naics_df`

- Using the same join keys as above, and treating the San Diego businesses as the left hand side data, left join the naics code details onto the San Diego businesses
- Use the `indicator` argument within merge to create an indicator, `naics_merge_status`, to help with later merge diagnostics and examine sample of ones that didn't merge
- Use the `suffixes` argument within merge to add `_sd` as the left suffix, `_census` as the right suffix
- Use `naics_merge_status` in the merged result to look at a sample of San Diego businesses that weren't matched with `naics_df`

In [43]:
left_merged_df = sd_df.merge(naics_df, left_on="naics_code", right_on="naics", how="left", indicator = "naics_merge_status", suffixes= ["_sd", "_census"])
left_merged_df.head()

,account_key,dba_name,council_district,naics_code,naics_description_sd,naics_nchar,naics,naics_description_census,naics_merge_status
0,1974000448,ERNST & YOUNG LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants,both
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,cd_3,5411,LEGAL SERVICES,4,NaN,NaN,left_only
2,1978039819,RSM US LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants,both
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,cd_3,5411,LEGAL SERVICES,4,NaN,NaN,left_only
4,1979046817,KORENIC & WOJDOWSKI LLP,cd_7,5412,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,4,NaN,NaN,left_only


In [51]:
left_merged_df[left_merged_df["naics_merge_status"] == "left_only"].sample(10)

,account_key,dba_name,council_district,naics_code,naics_description_sd,naics_nchar,naics,naics_description_census,naics_merge_status,is_lost
35,1998009401,DUE PROCESS,cd_6,5411,LEGAL SERVICES,4,NaN,NaN,left_only,True
290,2020000658,FF PROPERTIES II LP,cd_6,53131,REAL ESTATE PROPERTY MANAGERS,5,NaN,NaN,left_only,True
292,2020004240,BREMER WHYTE BROWN & OMEARA LLP,cd_3,54111,OFFICES OF LAWYERS,5,NaN,NaN,left_only,True
195,2015032777,CUCINA ENOTECA DEL MAR,cd_3,7221,FULL-SERVICE RESTAURANTS,4,NaN,NaN,left_only,True
332,2020011853,FORCEFIELD LLC,cd_6,541514,IT SERVICES & SUPPORT,6,NaN,NaN,left_only,True
266,2019015990,CLK 1 JUICE LP,cd_9,7222,LIMITED-SERVICE EATING PLACES,4,NaN,NaN,left_only,True
270,2019015995,CLK 2 JUICE LP,cd_5,7222,LIMITED-SERVICE EATING PLACES,4,NaN,NaN,left_only,True
10,1987006330,FINCH THORTON & BAIRD LLP,cd_1,5411,LEGAL SERVICES,4,NaN,NaN,left_only,True
112,2007007910,IV DAES LP,cd_7,5239,OTHER FINANCIAL INVESTMENT ACTIVITIES,4,NaN,NaN,left_only,True
168,2011034178,HKG DUTY FREE,cd_8,44531,"BEER, WINE & LIQUOR STORES",5,NaN,NaN,left_only,True


## Use group by and agg to see if there are differences in merge rates by area

- Using the left-joined dataframe created in previous step, create a boolean indicator, `is_lost`, that equals `True` if the merge indicator is equal to "left_only" (and is otherwise false)
- Find the proportion of businesses by council district that were lost in the left join (SD businesses that failed to match to `naics_df`). To do this, group by `council_district` and use the shortcut of taking the mean of the `is_lost` indicator to find the proportions lost by `council_district`. 

In [48]:
left_merged_df["is_lost"] = left_merged_df["naics_merge_status"] == "left_only"
left_merged_df.head()

,account_key,dba_name,council_district,naics_code,naics_description_sd,naics_nchar,naics,naics_description_census,naics_merge_status,is_lost
0,1974000448,ERNST & YOUNG LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants,both,False
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,cd_3,5411,LEGAL SERVICES,4,NaN,NaN,left_only,True
2,1978039819,RSM US LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants,both,False
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,cd_3,5411,LEGAL SERVICES,4,NaN,NaN,left_only,True
4,1979046817,KORENIC & WOJDOWSKI LLP,cd_7,5412,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,4,NaN,NaN,left_only,True


In [49]:
left_merged_df.groupby("council_district")["is_lost"].mean()

council_district
cd_1    0.880597
cd_2    0.804348
cd_3    0.812500
cd_4    1.000000
cd_5    0.916667
cd_6    0.859649
cd_7    0.918919
cd_8    0.933333
cd_9    1.000000
Name: is_lost, dtype: float64

## Optional challenge exercise: add lagging 0's and see if merge rate improves

You noticed earlier that a big reason for non-matches is that the San Diego tax certificate NAICS codes were often less than six digits long, while the Census ones were always 6 digits.

You wonder if this is an issue where 0's in some of the SD's data naics codes got cut off (eg 540000 became 54), and if so whether adding these lagging zeros would improve the merge rate in a left join.

- Pad the `naics_code` column in `sd_df` with 0's to get that column up to 6-digits, using one of two approaches: 
  1. `str.pad` in pandas (https://pandas.pydata.org/docs/reference/api/pandas.Series.str.pad.html)
  2. for more of a challenge, write your own function! It should check the # of digits in the naics code string and pad it with 0's at the end up to 6 digits. To execute your function, use row-wise apply: `df.apply(lambda row: funcname(row.column), axis=1)`.
- Perform the same left join as above and see how the match rate changes
- Create an indicator variable--`is_new_match`---for new matches under the padded NAICS code; compare the `naics_description` column from San Diego versus Census in the new dataset for a sample of these new matches and comment on whether the padding seems to be correct

In [52]:
def padzero(x):
    curr = str(x)
    while len(curr) < 6:
        curr += '0'

    return (curr)


SD["naics_code"] = SD.apply(lambda row: padzero(row.naics_code), axis = 1)

df3 = SD.merge(NA, 
               left_on = "naics_code", 
               right_on = "naics", 
               how="left", 
               indicator = 'naics_merge_status',
               suffixes = ['_sd', '_census']
              )
df3["is_new_match"] = df3["naics_merge_status"] == "left_only"

df3[["is_new_match", "naics_description_sd", "naics_description_census"]]

,is_new_match,naics_description_sd,naics_description_census
0,False,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,Offices of Certified Public Accountants
1,True,LEGAL SERVICES,NaN
2,False,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,Offices of Certified Public Accountants
3,True,LEGAL SERVICES,NaN
4,True,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,NaN
...,...,...,...
441,True,"PROFESSIONAL, SCIENTIFIC & TECHNICAL SERVICES",NaN
442,False,ALL OTHER AUTOMOTIVE R&M,All Other Automotive Repair and Maintenance
443,False,ALL OTHER AUTOMOTIVE R&M,All Other Automotive Repair and Maintenance
444,False,ALL OTHER AUTOMOTIVE R&M,All Other Automotive Repair and Maintenance


In [56]:
SD["padded_naics_code"] = SD["naics_code"].str.pad(6, side="right", fillchar = "0")
SD.head()

,account_key,dba_name,council_district,naics_code,naics_description,naics_nchar,padded_naics_code
0,1974000448,ERNST & YOUNG LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,cd_3,541100,LEGAL SERVICES,4,541100
2,1978039819,RSM US LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,cd_3,541100,LEGAL SERVICES,4,541100
4,1979046817,KORENIC & WOJDOWSKI LLP,cd_7,541200,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,4,541200
